# Chunk Visualization
This notebook compares raw Docling chunking with the updated `LayoutAwareChunker`.

In [1]:
import sys
import os
sys.path.append(os.path.abspath('../src'))

import pandas as pd
pd.set_option('display.max_colwidth', 150)
pd.set_option('display.max_rows', 100)

from arxiv_scholar.chunking.layout import LayoutAwareChunker
from arxiv_scholar.schema import Document
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.datamodel.pipeline_options import PdfPipelineOptions, AcceleratorOptions, AcceleratorDevice
from docling.datamodel.base_models import InputFormat
from docling.chunking import HierarchicalChunker


/Users/tri/Projects/arxiv-scholar/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [9]:
# 1. Load PDF using Docling
# Picked an existing PDF from the tests/notebooks directory
pdf_path = "../notebooks/notebook_trial/0704.0001v2.pdf"
if not os.path.exists(pdf_path):
    pdf_path = "../notebooks/notebook_trial/0704.0001v2.pdf"

pipeline_options = PdfPipelineOptions()
pipeline_options.do_ocr = True
pipeline_options.do_formula_enrichment = True
pipeline_options.accelerator_options = AcceleratorOptions(num_threads=1, device=AcceleratorDevice.CPU)

converter = DocumentConverter(
    format_options={InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_options)}
)

print(f"Converting {pdf_path}...")
dl_doc = converter.convert(pdf_path).document
print("Conversion complete.")


[INFO] 2026-06-05 11:38:59,467 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-06-05 11:38:59,481 [RapidOCR] download_file.py:60: File exists and is valid: /Users/tri/Projects/arxiv-scholar/.venv/lib/python3.10/site-packages/rapidocr/models/ch_PP-OCRv4_det_mobile.onnx
[INFO] 2026-06-05 11:38:59,481 [RapidOCR] main.py:57: Using /Users/tri/Projects/arxiv-scholar/.venv/lib/python3.10/site-packages/rapidocr/models/ch_PP-OCRv4_det_mobile.onnx


Converting ../notebooks/notebook_trial/0704.0001v2.pdf...


[INFO] 2026-06-05 11:38:59,564 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-06-05 11:38:59,567 [RapidOCR] download_file.py:60: File exists and is valid: /Users/tri/Projects/arxiv-scholar/.venv/lib/python3.10/site-packages/rapidocr/models/ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO] 2026-06-05 11:38:59,570 [RapidOCR] main.py:57: Using /Users/tri/Projects/arxiv-scholar/.venv/lib/python3.10/site-packages/rapidocr/models/ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO] 2026-06-05 11:38:59,596 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-06-05 11:38:59,605 [RapidOCR] download_file.py:60: File exists and is valid: /Users/tri/Projects/arxiv-scholar/.venv/lib/python3.10/site-packages/rapidocr/models/ch_PP-OCRv4_rec_mobile.onnx
[INFO] 2026-06-05 11:38:59,607 [RapidOCR] main.py:57: Using /Users/tri/Projects/arxiv-scholar/.venv/lib/python3.10/site-packages/rapidocr/models/ch_PP-OCRv4_rec_mobile.onnx
Loading weights: 100%|██████████| 770/770 [00:00<00:00, 11301.

Conversion complete.


### 1. Raw Docling Chunks
These are the micro-chunks output directly by Docling. Notice the very small text blocks.

In [10]:
hierarchical_chunker = HierarchicalChunker()
raw_chunks = list(hierarchical_chunker.chunk(dl_doc))

print(f"Total raw chunks: {len(raw_chunks)}")
raw_data = [{"Index": i, "Length": len(c.text), "Text": c.text} for i, c in enumerate(raw_chunks)]
df_raw = pd.DataFrame(raw_data)
display(df_raw.head(30))


Total raw chunks: 756


,Index,Length,Text
0,0,3599,/BV/BA /BU/CP/D0/G9/DE/D7 1 /B8 ∗ /BX/BA /BA /BU/CT/D6/CV/CT/D6 1 /B8 † /BA /CP/CS/D3/D0/D7/CZ/DD 1 /B8 ‡ /CP/D2/CS /BV/BA/B9 /BA /CH/D9/CP/D2...
1,1,3,∗ †
2,2,1,‡
3,3,1,§
4,4,55,/CQ/CP/D0/CP/DE/D7/CW/CT/D4/BA/CP/D2/D0/BA/CV/D3/DA/BN
5,5,40,/CQ/CT/D6/CV/CT/D6/CP/D2/D0/BA/CV/D3/DA
6,6,43,/DD/D9/CP/D2/D4/CP/BA/D1/D7/D9/BA/CT/CS/D9
7,7,21,/BV/D9/D6/D6/CT/D2/D8
8,8,58,/D2/CP/CS/D3/D0/D7/CZ/DD/CW/CT/D4/BA/CP/D2/D0/BA/CV/D3/DA
9,9,24,/CP/CS/CS/D6/CT/D7/D7/BM


### 2. LayoutAwareChunker Output
This runs our updated chunking logic which merges chunks up to `target_chunk_size` (1000 characters / ~200 words).

In [11]:
document = Document(
    id="test_doc_1",
    content="",
    metadata={"source_path": pdf_path}
)

chunker = LayoutAwareChunker(max_chunk_size=2000, target_chunk_size=1000)
processed_chunks = list(chunker.chunk(document))

print(f"Total merged chunks: {len(processed_chunks)}")
processed_data = [{"Index": c.metadata.get("chunk_index"), "Strategy": c.metadata.get("chunking_strategy"), "Length": len(c.content), "Text": c.content} for c in processed_chunks]
df_processed = pd.DataFrame(processed_data)
display(df_processed.head(30))


Loading weights: 100%|██████████| 770/770 [00:00<00:00, 8681.43it/s]


Total merged chunks: 151


,Index,Strategy,Length,Text
0,0,layout_aware_fallback,1999,/BV/BA /BU/CP/D0/G9/DE/D7 1 /B8 ∗ /BX/BA /BA /BU/CT/D6/CV/CT/D6 1 /B8 † /BA /CP/CS/D3/D0/D7/CZ/DD 1 /B8 ‡ /CP/D2/CS /BV/BA/B9 /BA /CH/D9/CP/D2...
1,1,layout_aware_fallback,1799,/D7/D4/CP\r/CT /CX/D7 /D7/D4/CT\r/CX/AS/CT/CS /CX/D2 /DB/CW/CX\r/CW /D8/CW/CT /CP/D0\r/D9/D0/CP/D8/CX/D3/D2 /CX/D7 /D1/D3/D7/D8 /D6/CT/D0/CX/CP/CQ...
2,2,layout_aware_fallback,1973,∗ †\n\n‡\n\n§\n\n/CQ/CP/D0/CP/DE/D7/CW/CT/D4/BA/CP/D2/D0/BA/CV/D3/DA/BN\n\n/CQ/CT/D6/CV/CT/D6/CP/D2/D0/BA/CV/D3/DA\n\n/DD/D9/CP/D2/D4/CP/BA/D1/...
3,3,layout_aware_fallback,840,/D8 /D1/CT/CP/D7/D9/D6/CT/D1/CT/D2/D8 /D3/CU  /CX/CV/CV/D7 /CQ/D3/D7/D3/D2 /D3/D9/D4/D0/CX/D2/CV /D7/D8/D6/CT/D2/CV/D8/CW/D7/BA /D2 /D8/CW/CX/D7 ...
4,4,layout_aware,1142,/D3/D4/CX/D3/D9/D7/D0/DD /D4/D6/D3/CS/D9\r/CT/CS /D2/D3/D2/B9/CX/D7/D3/D0/CP/D8/CT/CS /D4/CW/D3/D8/D3/D2/D7 /D8/CW/CP/D8 /CP/D6/CX/D7/CT /CU/D6/D3...
5,5,layout_aware_fallback,1998,γγ /CP /CU/CT/D6/D1/CX/D3/D2/B9/D0/D3/D3/D4 /CS/CX/CP/CV/D6/CP/D1/BA /BT/D8 /D0/D3/DB/CT/D7/D8 /D3/D6/CS/CT/D6 /CX/D2 /BV/BW/B8 /CP /D4/CW/D3/D8/D...
6,6,layout_aware_fallback,559,D1/D4/D9/D8/CT/CS /CX/D2 /CA/CT/CU/BA /CJ/BL/B8 /BD/BC℄/BA /D2 /D8/CW/CX/D7 /D7/D8/D9/CS/DD /DB/CT /CP/D0/D7/D3 /CX/D2\r/D0/D9/CS/CT /D7/D9/CQ/D0...
7,7,layout_aware_fallback,1996,/D8/D6/CX/CQ/D9/D8/CX/D3/D2/D7 /D8/D3 /CS/CX/D4/CW/D3/D8/D3/D2 /D4/D6/D3/CS/D9\r/D8/CX/D3/D2 /CU/D6/D3/D1 /D4/CP/D6/D8/D3/D2/B9/D4/CP/D6/D8/D3/D2 ...
8,8,layout_aware_fallback,1982,/D0/DD/B9/CT/D2/CW/CP/D2\r/CT/CS /D3/D2/D8/D6/CX/CQ/D9/D8/CX/D3/D2/D7/B8 /CP/D0/D7/D3 /CZ/D2/D3/DB/D2 /CP/D7 /CO/CU/D6/CP/CV/D1/CT/D2/D8/CP/D8/CX/...
9,9,layout_aware_fallback,388,6/D3/D1 /CP/D4/D4/D6/CT\r/CX/CP/CQ/D0/CT /CW/CP/CS/D6/D3/D2/CX /D6/CT/D1/D2/CP/D2/D8/D7 /CX/D2 /D8/CW/CT /D4/CP/D6/D8/CX\r/D0/CT /CS/CT/D8/CT\r/D8...
